# Notebook 3: Model Training and Validation
- Define data augmentations for training
- Implement fine-tuned ResNet18 model with pretrained weights
- Set up loss with class weights to handle imbalance
- Train model with progress bars and validation accuracy tracking
- Save best model checkpoint based on validation performance

In [1]:
# Import ibraries
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
import random
from torchvision.models import resnet18, ResNet18_Weights

# Set random seed to make results reproducible (same results every time)
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed()

# Setup device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define class labels for skin lesions (7 classes)
class_columns = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']

# Load training and validation CSV files with image paths and labels
train_df = pd.read_csv("train_df_processed.csv")
val_df = pd.read_csv("val_df_processed.csv")

# Define image transformations for training data (with augmentations)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),          # Randomly flip images horizontally
    transforms.RandomRotation(15),              # Randomly rotate images by 15 degrees
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # Randomly change brightness & contrast
    transforms.ToTensor(),                       # Convert images to PyTorch tensors
    transforms.Normalize([0.485, 0.456, 0.406], # Normalize with ImageNet mean
                         [0.229, 0.224, 0.225]) # Normalize with ImageNet std
])

# Define image transformations for validation data (no augmentation)
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Create a custom dataset class for loading skin lesion images and labels
class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_path = self.data.iloc[idx]['image_path']
        label = self.data.iloc[idx]['label_idx']
        image = Image.open(image_path).convert("RGB")  # Open image and convert to RGB
        if self.transform:
            image = self.transform(image)  # Apply transformations if there are any
        return image, label

# Create dataset instances for training and validation
train_ds = SkinLesionDataset(train_df, transform=train_transform)
val_ds = SkinLesionDataset(val_df, transform=val_transform)

# Create data loaders to load data in batches during training and validation
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=0)

# Load pretrained ResNet18 model with ImageNet weights
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# Freeze all layers except last two (layer4 and fully connected layer)
for name, param in model.named_parameters():
    if "layer4" in name or "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# Replace the final layer to output predictions for 7 classes
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(class_columns))

# Move the model to the device being used
model = model.to(device)

# Calculate class weights to handle class imbalance in training data
class_weights_np = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label_idx']),
    y=train_df['label_idx']
)
class_weights = torch.tensor(class_weights_np, dtype=torch.float).to(device)

# Define loss function with class weights to penalize imbalanced classes
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Define optimizer (Adam) only for parameters that require gradients
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# Define learning rate scheduler to reduce learning rate every fifth epochs
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# Define function to train the model for given number of epochs
def train_model(model, criterion, optimizer, train_loader, val_loader, scheduler=None, epochs=10):
    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()  # Set model to training mode
        total_loss = 0.0
        correct = 0
        total = 0

        # tqdm progress bar for training batches
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)

        for images, labels in train_bar:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()          
            outputs = model(images)  
            loss = criterion(outputs, labels) 
            loss.backward()           
            optimizer.step()      

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)  
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            # Update progress bar with current loss and accuracy
            train_bar.set_postfix(loss=loss.item(), accuracy=correct/total)

        train_acc = correct / total
        avg_loss = total_loss / len(train_loader)

        if scheduler:
            scheduler.step()  # Update learning rate

        # Evaluate on validation set
        val_acc = evaluate_model(model, val_loader, class_columns, print_report=False)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

        # Save model if validation accuracy improved
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_resnet18_model.pth")
            print(f"Best model saved with val acc: {best_val_acc:.4f}")

    print(f"Training complete. Best val acc: {best_val_acc:.4f}")

# Define function to evaluate model on validation data
def evaluate_model(model, val_loader, class_names, print_report=True):
    model.eval() 
    val_correct = 0
    val_total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc="Validation", leave=False)
        for images, labels in val_bar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            val_bar.set_postfix(val_accuracy=val_correct/val_total)

    val_acc = val_correct / val_total

    if print_report:
        print(f"Validation Accuracy: {val_acc:.4f}")
        print("\nClassification Report:")
        print(classification_report(all_labels, all_preds, target_names=class_names))

    return val_acc

# Train the model for 5 epochs (change epochs as needed)
train_model(model, criterion, optimizer, train_loader, val_loader, scheduler, epochs=5)

# Load the best saved model and evaluate on validation data
model.load_state_dict(torch.load("best_resnet18_model.pth"))
evaluate_model(model, val_loader, class_columns)


Using device: cpu


Epoch 1/5 | Train Loss: 1.1983 | Train Acc: 0.6093 | Val Acc: 0.6545
Best model saved with val acc: 0.6545


Epoch 2/5 | Train Loss: 0.8695 | Train Acc: 0.6947 | Val Acc: 0.6885
Best model saved with val acc: 0.6885


Epoch 3/5 | Train Loss: 0.7424 | Train Acc: 0.7222 | Val Acc: 0.7399
Best model saved with val acc: 0.7399


Epoch 4/5 | Train Loss: 0.6336 | Train Acc: 0.7484 | Val Acc: 0.7599
Best model saved with val acc: 0.7599


Epoch 5/5 | Train Loss: 0.5825 | Train Acc: 0.7619 | Val Acc: 0.7354
Training complete. Best val acc: 0.7599


Validation Accuracy: 0.7599

Classification Report:
              precision    recall  f1-score   support

         MEL       0.40      0.68      0.50       223
          NV       0.95      0.79      0.86      1341
         BCC       0.61      0.88      0.73       103
       AKIEC       0.45      0.69      0.55        65
         BKL       0.66      0.58      0.62       220
          DF       0.73      0.70      0.71        23
        VASC       0.65      0.93      0.76        28

    accuracy                           0.76      2003
   macro avg       0.64      0.75      0.68      2003
weighted avg       0.82      0.76      0.78      2003



0.7598602096854717